# Lab | Data Structuring and Combining Data

In [ ]:
import pandas as pd
import numpy as np


## Challenge 1: Combining & Cleaning Data

We load the three files, then apply a single cleaning function to each of them before concatenating. Cleaning each file separately (rather than concatenating first) makes it easier to catch differences between files, since the three files don't share exactly the same column names or ordering.

In [ ]:
url1 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv'
url2 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv'
url3 = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv'

df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)

# Always inspect before combining: the 3 files don't necessarily share
# the same column names or the same column order
print(df1.columns.tolist())
print(df2.columns.tolist())
print(df3.columns.tolist())


`file1` and `file2`they both use `ST` for the state column, while `file3` already uses `State`. The cleaning function below normalizes this but also every other inconsistencies. so we finally have in one place the harmonizing that can be applied identically to all three files.

In [ ]:
def clean_column_names(df):
    df = df.copy()
    df.columns = (df.columns
                    .str.strip()
                    .str.lower()
                    .str.replace(' ', '_'))
    df = df.rename(columns={'st': 'state'})
    return df


def clean_complaints(val):
    """'Number of Open Complaints' shows up either as an integer (file3)
    or as a string like '1/2/00' (file1, file2), where the middle
    number is the actual complaint count."""
    if pd.isna(val):
        return np.nan
    val = str(val)
    if '/' in val:
        return float(val.split('/')[1])
    return float(val)


def clean_data(df):
    df = clean_column_names(df)

    # Gender: standardize to 'F' / 'M'
    if 'gender' in df.columns:
        df['gender'] = df['gender'].replace({
            'Femal': 'F', 'Female': 'F', 'female': 'F',
            'Male': 'M', 'male': 'M'
        })

    # State: expand abbreviations to full names
    if 'state' in df.columns:
        state_mapping = {'AZ': 'Arizona', 'Cali': 'California', 'WA': 'Washington'}
        df['state'] = df['state'].replace(state_mapping)

    # Customer Lifetime Value: remove '%' and convert to float
    if 'customer_lifetime_value' in df.columns:
        df['customer_lifetime_value'] = (
            df['customer_lifetime_value']
            .astype(str)
            .str.replace('%', '', regex=False)
            .astype(float)
        )

    # Vehicle class: group luxury categories together
    if 'vehicle_class' in df.columns:
        df['vehicle_class'] = df['vehicle_class'].replace({
            'Sports Car': 'Luxury',
            'Luxury SUV': 'Luxury',
            'Luxury Car': 'Luxury'
        })

    # Number of open complaints: keep only the actual complaint count
    if 'number_of_open_complaints' in df.columns:
        df['number_of_open_complaints'] = df['number_of_open_complaints'].apply(clean_complaints)

    # Drop fully empty rows and exact duplicates
    df = df.dropna(how='all')
    df = df.drop_duplicates()

    return df


In [ ]:
df1_clean = clean_data(df1)
df2_clean = clean_data(df2)
df3_clean = clean_data(df3)

df_combined = pd.concat([df1_clean, df2_clean, df3_clean], ignore_index=True)
df_combined = df_combined.drop_duplicates()

print(df_combined.shape)
df_combined.head()


In [ ]:
df_combined.isna().sum()


**Conclusion — Challenge 1:** After cleaning and standardizing the three datasets, we successfully consolidated them into a single consistent dataset containing 9,134 rows and 11 columns. Column names were harmonized, particularly ST and State, which were standardized to state. We also cleaned and standardized key categorical variables such as gender, state, and vehicle_class, while converting customer_lifetime_value and number_of_open_complaints into appropriate formats.

Only a small number of missing values remain, mainly in gender (122 values) and customer_lifetime_value (7 values), along with a few missing values in number_of_open_complaints. These missing values can easily be addressed during the next stages through appropriate imputation or removal strategies.

After standardization, the dataset contains 5 states — California, Oregon, Arizona, Nevada, and Washington — and 4 vehicle classes: Four-Door Car, Two-Door Car, SUV, and Luxury. Overall, the cleaning process confirms that the three original files represent the same customer population, with consistent categories after harmonization.

## Challenge 2: Structuring Data

We load `marketing_customer_analysis_clean.csv`, which is already cleaned, and focus on `pivot_table()` — the natural next step after `groupby()`.

In [ ]:
url_marketing = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv'
df_marketing = pd.read_csv(url_marketing)

if 'unnamed:_0' in df_marketing.columns:
    df_marketing = df_marketing.drop(columns=['unnamed:_0'])

df_marketing.head()


### 1. Total revenue by sales channel

`total_claim_amount` is used as the proxy for revenue generated, summed by `sales_channel`, as in the dataset there is no explicit "revenue" column.

In [ ]:
revenue_by_channel = pd.pivot_table(
    df_marketing,
    values='total_claim_amount',
    index='sales_channel',
    aggfunc='sum'
).round(2)

revenue_by_channel.sort_values('total_claim_amount', ascending=False)


**Insight:**
The `Agent` channel generates the highest total claim amount, reaching **$1,810,226.82**, followed by `Branch` with **$1,301,204.00** and `Call Center` with **$926,600.82**. The `Web` channel records the lowest amount at **$706,600.04**.

The difference between `Agent` and `Web` exceeds **$1.1 million**, highlighting a significant performance gap between advisor-led and digital self-service channels. This result suggests that customers currently generate more business through direct or assisted interactions. The lower performance of the `Web` channel could therefore be further investigated to identify opportunities for improving digital engagement, customer retention, and online conversion.


### 2. Average Customer Lifetime Value by gender and education level

In [ ]:
clv_by_gender_education = pd.pivot_table(
    df_marketing,
    values='customer_lifetime_value',
    index='gender',
    columns='education',
    aggfunc='mean'
).round(2)

clv_by_gender_education


**Insight:**
Average Customer Lifetime Value (CLV) remains relatively consistent across education levels for both men and women, ranging from approximately **$7,300 to $8,700**. This suggests that education level alone does not appear to be a strong differentiating factor between high- and low-value customers.

The highest average CLV is observed among **women with a High School education or below**, at **$8,675.22**, followed by **men with the same education level** at **$8,149.69** and **men holding a Master's degree** at **$8,168.83**. In contrast, the lowest average CLV is found among customers with a **Doctorate degree**, with **$7,328.51 for women** and **$7,415.33 for men**.

The differences between men and women within the same education category are relatively small, generally remaining below **$400**. Overall, these results indicate that **gender and education have a limited relationship with CLV**, suggesting that other factors, such as **income, number of policies, or customer characteristics**, may play a more significant role in determining customer value.


### Bonus: number of complaints by policy type and month, in long format

We first build a wide table with `pivot_table`, then reshape it into long format with `melt()`: each row becomes a single observation of (`policy_type`, `month`, `number_of_complaints`).

In [ ]:
month_map = {1: 'January', 2: 'February'}
df_marketing['month_name'] = df_marketing['month'].map(month_map)

complaints_wide = pd.pivot_table(
    df_marketing,
    values='number_of_open_complaints',
    index='policy_type',
    columns='month_name',
    aggfunc='sum'
)

complaints_wide


In [ ]:
complaints_long = (
    complaints_wide
    .reset_index()
    .melt(id_vars='policy_type', var_name='month', value_name='number_of_complaints')
)

complaints_long.sort_values(['policy_type', 'month']).reset_index(drop=True)


**Conclusion — Bonus:**
The `complaints_long` dataset follows a proper **long-format structure**, with one column for each variable (`policy_type`, `month`, and `number_of_complaints`) and one row per observation. In contrast, `complaints_wide` presents the months as separate columns.

Across both months, **Personal Auto** records the highest number of complaints, with **1,727.6 in January** and **1,453.7 in February**. This is consistent with Personal Auto being the largest policy category overall. It is followed by **Corporate Auto**, with **443.4 complaints in January** and **385.2 in February**, while **Special Auto** has the lowest complaint levels, with **87.1 in January** and **95.2 in February**.

An interesting point is that **Special Auto is the only category showing a slight increase in complaints from January to February**. Meanwhile, both Personal Auto and Corporate Auto show higher complaint levels in January than in February. This may indicate a **moderate seasonal pattern**, although additional months of data would be needed to confirm whether this trend is consistent.
